IMPORTING LIBRARIES

In [3]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import KNNImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import shap
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

DATA LOADING AND PREPROCESSING


In [4]:
df_train = pd.read_csv('df_labeled_train.csv', encoding='latin1')
df_test  = pd.read_csv('df_labeled_test.csv',  encoding='latin1')


problem_cols = [
    'Account receivables / Account payables',
    'Long term liabilities / Total liabilities'
]

for col in problem_cols:
    for df in [df_train, df_test]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

KNN IMPUTATION

In [5]:
X_train_full = df_train.drop("target", axis=1)
X_test_full  = df_test.drop("target",  axis=1)
y_train_original = df_train["target"]
y_test_original  = df_test["target"]

imputer = KNNImputer(n_neighbors=5)
X_train_full = pd.DataFrame(
    imputer.fit_transform(X_train_full),
    columns=X_train_full.columns
)
X_test_full = pd.DataFrame(
    imputer.transform(X_test_full),
    columns=X_test_full.columns
)


UTILITY FUNCTIONS


In [6]:
def class_ratio(y):
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    return neg / pos


def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_pred, dtrain):
        y_true = dtrain.get_label()
        p    = 1.0 / (1.0 + np.exp(-y_pred))
        grad = alpha * (y_true - p) * ((1 - p) ** gamma)
        hess = alpha * ((1 - p) ** gamma) * (p * (1 - p))
        return -grad, hess
    return loss


def get_model_prob(model_dict, X):
    mtype = model_dict["type"]
    if mtype == "xgb":
        return model_dict["model"].predict(xgb.DMatrix(X))
    elif mtype == "lgb":
        return model_dict["model"].predict(X)
    elif mtype in ["cat", "sklearn"]:
        return model_dict["model"].predict_proba(X)[:, 1]

THRESHOLD CALIBRATION

In [7]:
def find_f1_optimal_threshold(X, y, model_fn, n_splits=5):
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_probs  = np.zeros(len(y))
    oof_labels = np.zeros(len(y))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model_dict = model_fn(X_tr, y_tr)
        oof_probs[val_idx]  = get_model_prob(model_dict, X_val)
        oof_labels[val_idx] = y_val.values

    precision_arr, recall_arr, thresh_arr = precision_recall_curve(
        oof_labels, oof_probs
    )

   
    f1_scores = (
        2 * precision_arr[:-1] * recall_arr[:-1]
        / np.maximum(precision_arr[:-1] + recall_arr[:-1], 1e-9)
    )

    best_idx       = np.argmax(f1_scores)
    best_threshold = thresh_arr[best_idx]
    best_f1        = f1_scores[best_idx]
    best_precision = precision_arr[best_idx]
    best_recall    = recall_arr[best_idx]

    print(f"  OOF F1-optimal threshold : {best_threshold:.4f}")
    print(f"  OOF F1                   : {best_f1:.4f}")
    print(f"  OOF Precision            : {best_precision:.4f}")
    print(f"  OOF Recall               : {best_recall:.4f}")

    return best_threshold

MODEL ZOO


In [8]:
def train_all_models(X_tr, y_tr, X_te, y_te, spw):
    results = {}

    # 1. XGBoost with focal loss
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dtest  = xgb.DMatrix(X_te, label=y_te)
    params_xgb = {
        "max_depth"        : 5,
        "eta"              : 0.05,
        "subsample"        : 0.8,
        "colsample_bytree" : 0.8,
        "objective"        : "binary:logistic",
        "eval_metric"      : "logloss",
        "scale_pos_weight" : spw,
        "seed"             : 42
    }
    xgb_model = xgb.train(
        params_xgb, dtrain, num_boost_round=300,
        obj=focal_loss(alpha=0.25, gamma=2.0)
    )
    prob_xgb = xgb_model.predict(dtest)
    results["XGBoost"] = {
        "model": xgb_model, "proba": prob_xgb,
        "pr_auc": average_precision_score(y_te, prob_xgb), "type": "xgb"
    }

    # 2. LightGBM
    lgb_train = lgb.Dataset(X_tr, label=y_tr)
    params_lgb = {
        "objective"        : "binary",
        "metric"           : "average_precision",
        "scale_pos_weight" : spw,
        "max_depth"        : 5,
        "learning_rate"    : 0.01,
        "num_leaves"       : 31,
        "subsample"        : 0.8,
        "colsample_bytree" : 0.8,
        "seed"             : 42,
        "verbose"          : -1
    }
    lgb_model = lgb.train(params_lgb, lgb_train, num_boost_round=300)
    prob_lgb  = lgb_model.predict(X_te)
    results["LightGBM"] = {
        "model": lgb_model, "proba": prob_lgb,
        "pr_auc": average_precision_score(y_te, prob_lgb), "type": "lgb"
    }

    # 3. CatBoost
    cat_model = CatBoostClassifier(
        iterations=300, depth=5, learning_rate=0.01,
        class_weights=[1.0, spw], eval_metric="AUC",
        random_seed=42, verbose=0
    )
    cat_model.fit(X_tr, y_tr)
    prob_cat = cat_model.predict_proba(X_te)[:, 1]
    results["CatBoost"] = {
        "model": cat_model, "proba": prob_cat,
        "pr_auc": average_precision_score(y_te, prob_cat), "type": "cat"
    }

    # 4. Random Forest
    rf_model = RandomForestClassifier(
        n_estimators=300, max_depth=10, class_weight="balanced",
        random_state=42, n_jobs=-1
    )
    rf_model.fit(X_tr, y_tr)
    prob_rf = rf_model.predict_proba(X_te)[:, 1]
    results["RandomForest"] = {
        "model": rf_model, "proba": prob_rf,
        "pr_auc": average_precision_score(y_te, prob_rf), "type": "sklearn"
    }

    # 5. Logistic Regression
    lr_model = LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    )
    lr_model.fit(X_tr, y_tr)
    prob_lr = lr_model.predict_proba(X_te)[:, 1]
    results["LogisticRegression"] = {
        "model": lr_model, "proba": prob_lr,
        "pr_auc": average_precision_score(y_te, prob_lr), "type": "sklearn"
    }

    return results


STAGE 1- DEFAULT VS NO DEFAULT

In [9]:
print("\n===================================================")
print("STAGE 1 : DEFAULT PREDICTION")
print("===================================================")

y_train_stage1 = (y_train_original > 0).astype(int)
y_test_stage1  = (y_test_original  > 0).astype(int)

# SHAP feature selection on full training set
base_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    eval_metric='logloss', random_state=42
)
base_model.fit(X_train_full, y_train_stage1)
explainer   = shap.TreeExplainer(base_model)
shap_values = explainer.shap_values(X_train_full)

if isinstance(shap_values, list):
    shap_vals = np.array(shap_values).mean(axis=0)
elif len(shap_values.shape) == 3:
    shap_vals = shap_values.mean(axis=2)
else:
    shap_vals = shap_values

importance        = np.abs(shap_vals).mean(axis=0)
feature_importance = pd.Series(importance, index=X_train_full.columns)
top_features       = feature_importance.sort_values(ascending=False).head(20).index

print("\nTop Features:")
print(top_features.tolist())

X_train_stage1 = X_train_full[top_features]
X_test_stage1  = X_test_full[top_features]
spw = class_ratio(y_train_stage1)
print(f"\nClass ratio (neg/pos) : {spw:.2f}")



print("\n--- Stage 1 Threshold Calibration (Stratified 5-Fold OOF) ---")

def build_catboost_s1(X_tr, y_tr):
    spw_fold = class_ratio(y_tr)
    m = CatBoostClassifier(
        iterations=300, depth=5, learning_rate=0.01,
        class_weights=[1.0, spw_fold], eval_metric="AUC",
        random_seed=42, verbose=0
    )
    m.fit(X_tr, y_tr)
    return {"model": m, "type": "cat"}

best_threshold = find_f1_optimal_threshold(
    X_train_stage1, y_train_stage1, build_catboost_s1, n_splits=5
)

# Train final Stage 1 models on all training data and pick best by PR-AUC
stage1_results = train_all_models(
    X_train_stage1, y_train_stage1, X_test_stage1, y_test_stage1, spw
)

print("\n--- Stage 1 PR-AUC Scores (full train / test) ---")
for name, res in stage1_results.items():
    print(f"  {name:<22} : {res['pr_auc']:.4f}")

best_s1_name = max(stage1_results, key=lambda k: stage1_results[k]["pr_auc"])
best_s1      = stage1_results[best_s1_name]
print(f"\nBest Stage 1 Model : {best_s1_name}  (PR-AUC = {best_s1['pr_auc']:.4f})")

y_pred_prob_stage1 = best_s1["proba"]

# Report what the chosen threshold yields on the test set (informational only)
y_pred_stage1 = (y_pred_prob_stage1 >= best_threshold).astype(int)
test_f1 = f1_score(y_test_stage1, y_pred_stage1)
print(f"\nThreshold used (OOF F1-optimal) : {best_threshold:.4f}")
print(f"Test-set F1 at this threshold    : {test_f1:.4f}")



STAGE 1 : DEFAULT PREDICTION

Top Features:
['t-1 TFA', '(equity + total income) / Total assets', 'Provision for Tax', 'CBS_CUSTOM', 'Cash to total debt', 'I. Term Loan from Our Bank', 'Others', 'Total Debtors_22', 'Inventories growth / Sales growth', 'Total Other Long Term Liabilities', 'year', 'Assets_prev_year', 'Fixed asset turnover', 'III. General Reserves (including Credit Balance in P&L )', 'Interest / Turnover', 'II. Plant & Machinery', 'Borrowings from Our Bank', 'Inventories growth', 'Cash flow / Total debt', 'ii. From institutions/Banks-2']

Class ratio (neg/pos) : 17.82

--- Stage 1 Threshold Calibration (Stratified 5-Fold OOF) ---
  OOF F1-optimal threshold : 0.5325
  OOF F1                   : 0.2903
  OOF Precision            : 0.2174
  OOF Recall               : 0.4369

--- Stage 1 PR-AUC Scores (full train / test) ---
  XGBoost                : 0.2776
  LightGBM               : 0.2148
  CatBoost               : 0.3482
  RandomForest           : 0.2778
  LogisticRegres

STAGE 2- MILD VS SEVERE

In [10]:
print("\n===================================================")
print("STAGE 2 : SEVERITY PREDICTION")
print("===================================================")

train_defaults = df_train[df_train["target"] > 0].copy()
test_defaults  = df_test[df_test["target"]  > 0].copy()

def severity_map(x):
    return 0 if x == 1 else 1

train_defaults["severity"] = train_defaults["target"].apply(severity_map)
test_defaults["severity"]  = test_defaults["target"].apply(severity_map)

X_train_stage2 = train_defaults.drop(["target", "severity"], axis=1)
X_test_stage2  = test_defaults.drop(["target", "severity"],  axis=1)
y_train_stage2 = train_defaults["severity"]
y_test_stage2  = test_defaults["severity"]

for col in problem_cols:
    for df in [X_train_stage2, X_test_stage2]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# Impute on defaulters sub-population only to respect their distribution
imputer_s2 = KNNImputer(n_neighbors=5)
X_train_stage2 = pd.DataFrame(
    imputer_s2.fit_transform(X_train_stage2),
    columns=X_train_stage2.columns
)
X_test_stage2 = pd.DataFrame(
    imputer_s2.transform(X_test_stage2),
    columns=X_test_stage2.columns
)

common_features = [f for f in top_features if f in X_train_stage2.columns]
X_train_stage2 = X_train_stage2[common_features].copy()
X_test_stage2  = X_test_stage2[common_features].copy()

# Add Stage 1 probability as feature
s1_train_feats = X_train_full.loc[train_defaults.index, top_features]
s1_test_feats  = X_test_full.loc[test_defaults.index,  top_features]

X_train_stage2['s1_prob'] = get_model_prob(best_s1, s1_train_feats)
X_test_stage2['s1_prob']  = get_model_prob(best_s1, s1_test_feats)
common_features_s2 = list(X_train_stage2.columns)

spw2 = class_ratio(y_train_stage2)
print(f"Stage 2 Class ratio (mild/severe) : {spw2:.2f}")


print("\n--- Stage 2 Threshold Calibration (Stratified 5-Fold OOF) ---")

def build_logreg_s2(X_tr, y_tr):
    m = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
    m.fit(X_tr, y_tr)
    return {"model": m, "type": "sklearn"}

best_threshold_s2 = find_f1_optimal_threshold(
    X_train_stage2, y_train_stage2, build_logreg_s2, n_splits=5
)

stage2_results = train_all_models(
    X_train_stage2, y_train_stage2, X_test_stage2, y_test_stage2, spw2
)

print("\n--- Stage 2 PR-AUC Scores ---")
for name, res in stage2_results.items():
    print(f"  {name:<22} : {res['pr_auc']:.4f}")

best_s2_name = max(stage2_results, key=lambda k: stage2_results[k]["pr_auc"])
best_s2      = stage2_results[best_s2_name]
print(f"\nBest Stage 2 Model : {best_s2_name}  (PR-AUC = {best_s2['pr_auc']:.4f})")

y_pred_prob_stage2 = best_s2["proba"]
y_pred_stage2      = (y_pred_prob_stage2 >= best_threshold_s2).astype(int)

stage2_accuracy = accuracy_score(y_test_stage2, y_pred_stage2)
stage2_f1       = f1_score(y_test_stage2, y_pred_stage2)
print(f"\nStage 2 Accuracy : {stage2_accuracy:.4f}")
print(f"Stage 2 F1 Score : {stage2_f1:.4f}")


STAGE 2 : SEVERITY PREDICTION
Stage 2 Class ratio (mild/severe) : 1.71

--- Stage 2 Threshold Calibration (Stratified 5-Fold OOF) ---
  OOF F1-optimal threshold : 0.2505
  OOF F1                   : 0.5926
  OOF Precision            : 0.4571
  OOF Recall               : 0.8421

--- Stage 2 PR-AUC Scores ---
  XGBoost                : 0.6551
  LightGBM               : 0.5213
  CatBoost               : 0.6600
  RandomForest           : 0.5549
  LogisticRegression     : 0.6653

Best Stage 2 Model : LogisticRegression  (PR-AUC = 0.6653)

Stage 2 Accuracy : 0.6000
Stage 2 F1 Score : 0.6316


FINAL PREDICTIONS

In [11]:
final_predictions = []

for i in range(len(X_test_full)):

    if y_pred_prob_stage1[i] < best_threshold:
        final_predictions.append(0)

    else:
        row_s1 = X_test_full.iloc[[i]][top_features]
        p1     = get_model_prob(best_s1, row_s1)[0]

        row_s2          = X_test_full.iloc[[i]][common_features].copy()
        row_s2['s1_prob'] = p1

        sev_prob = get_model_prob(best_s2, row_s2)[0]

        if sev_prob < best_threshold_s2:
            final_predictions.append(1)   # Mild
        else:
            final_predictions.append(2)   # Severe


def final_target_map(x):
    if x == 0:    return 0
    elif x == 1:  return 1
    else:         return 2

y_test_final = y_test_original.apply(final_target_map)

FINAL METRICS


In [12]:
print("\n===================================================")
print("FINAL HIERARCHICAL MODEL")
print("===================================================")

final_accuracy = accuracy_score(y_test_final, final_predictions)
final_macro_f1 = f1_score(y_test_final, final_predictions, average='macro')

print(f"\nFinal Accuracy : {final_accuracy:.4f}")
print(f"Final Macro F1 : {final_macro_f1:.4f}")

print("\nClassification Report\n")
print(classification_report(
    y_test_final,
    final_predictions,
    target_names=["No Default", "Mild Default", "Severe Default"]
))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test_final, final_predictions))



FINAL HIERARCHICAL MODEL

Final Accuracy : 0.8854
Final Macro F1 : 0.4532

Classification Report

                precision    recall  f1-score   support

    No Default       0.97      0.92      0.94       611
  Mild Default       0.28      0.23      0.25        22
Severe Default       0.11      0.38      0.17        13

      accuracy                           0.89       646
     macro avg       0.45      0.51      0.45       646
  weighted avg       0.93      0.89      0.90       646


Confusion Matrix

[[562  13  36]
 [ 11   5   6]
 [  8   0   5]]
